# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [31]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [32]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [33]:
# Task 1 — Multi-series line with highlight

df_asia = df[df['Region'] == 'Asia']
countries = df_asia['Country'].unique()
highlight_country = "China"

fig = go.Figure()

for country in countries:
    country_data = df_asia[df_asia['Country'] == country]
    is_highlight = (country == highlight_country)
    
    fig.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data['CO2_Mt'],
        # FIX: Combine modes in a single string
        mode='lines+text' if is_highlight else 'lines',
        name=country,
        line=dict(
            color='royalblue' if is_highlight else '#DDDDDD',
            width=3 if is_highlight else 1
        ),
        text=[None] * (len(country_data)-1) + [f"  {country}"] if is_highlight else None,
        textposition="middle right",
        hoverinfo='name+y'
    ))

fig.update_layout(
    title="<b>China's CO2 Emissions Surged Post-2000, Decoupling from Regional Peers</b><br>Annual CO2 emissions (Mt) for Asian countries, 2000–2022",
    font=dict(family="Arial", size=14, color="black"),
    paper_bgcolor='white',
    plot_bgcolor='white',
    showlegend=False,
    xaxis=dict(showgrid=False, range=[2000, 2026]),
    yaxis=dict(showgrid=False, title="CO2 Emissions (Mt)"),
    margin=dict(l=50, r=120, t=80, b=50)
)

fig.show()

---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [34]:
# Task 2 — Slopegraph: regional averages

df_regional = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()
df_slope = df_regional[df_regional['Year'].isin([2000, 2022])]

fig = go.Figure()

regions = df_slope['Region'].unique()

for region in regions:
    region_data = df_slope[df_slope['Region'] == region].sort_values('Year')
    val_2000 = region_data.iloc[0]['CO2_Mt']
    val_2022 = region_data.iloc[1]['CO2_Mt']
    
    line_color = "firebrick" if val_2022 > val_2000 else "seagreen"
    
    fig.add_trace(go.Scatter(
        x=region_data['Year'],
        y=region_data['CO2_Mt'],
        # FIX: markers and text combined in mode
        mode='lines+markers+text',
        name=region,
        line=dict(color=line_color, width=2),
        marker=dict(size=8),
        text=[f"{region} {val_2000:.0f} ", f" {val_2022:.0f}"],
        textposition=["middle left", "middle right"],
        hoverinfo='none'
    ))

fig.update_layout(
    title="<b>Asia and Africa Saw Emission Growth While Other Regions Declined</b><br>Average regional CO2 emissions (Mt), 2000 vs 2022",
    font=dict(family="Arial", size=14, color="black"),
    paper_bgcolor='white',
    plot_bgcolor='white',
    showlegend=False,
    xaxis=dict(tickvals=[2000, 2022], range=[1990, 2032], showgrid=False, showline=False),
    yaxis=dict(showticklabels=False, showgrid=False, showline=False),
    margin=dict(l=150, r=150, t=100, b=50)
)

fig.show()